In [8]:
import os
import shutil as sh
import pandas as pd
from src.vasijas.reporting import ProcessingReport
from _bootstrap import setup

PATHS = setup()
report = ProcessingReport()

archivo_cvs = PATHS.DATA_UBICADOS / "metadata.csv"

# Nombres de columnas en el CSV
COL_ID = "Identificador"
COL_CLASE = "Clase Ana"
COL_UBICACION = "Ubicacion"

# Leer el CSV
df = pd.read_csv(archivo_cvs)
df.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla


In [10]:
# Funciones
def nombre_carpeta_clase(valor_clase: str|int|float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'Clase01', 'Clase02', ... 'Clase11', etc.
    - Si está vacío/NaN → 'sin clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"

In [ ]:
archivos_copiados = 0
archivos_no_encontrados = 0
conteo_por_clase = {}  # nuevo: conteo de archivos copiados por carpeta de clase

for _, fila in df.iterrows():
    identificador = str(fila[COL_ID]).strip()
    clase_ana = fila[COL_CLASE]
    ubicacion = str(fila[COL_UBICACION]).strip()

    # Carpeta destino según clase
    carpeta_clase = nombre_carpeta_clase(clase_ana)
    ruta_destino_carpeta = PATHS.DATA_CLASIFICADOS / carpeta_clase
    os.makedirs(ruta_destino_carpeta, exist_ok=True)

    # Rutas origen y destino del archivo PNG
    nombre_png = f"{identificador}.png"
    ruta_origen = os.path.join(RUTA_ORIGINAL, ubicacion, nombre_png)
    ruta_destino = os.path.join(ruta_destino_carpeta, nombre_png)

    if os.path.exists(ruta_origen):
        sh.copy2(ruta_origen, ruta_destino)
        archivos_copiados += 1

        # Actualizar conteo por clase
        conteo_por_clase[carpeta_clase] = conteo_por_clase.get(carpeta_clase, 0) + 1
    else:
        print(f"No se encontró el archivo: {ruta_origen}")
        archivos_no_encontrados += 1

print(f"\nArchivos copiados: {archivos_copiados}")
print(f"Archivos no encontrados: {archivos_no_encontrados}\n")

# Mostrar resumen por clase, ordenado por nombre de carpeta
print("Resumen de archivos copiados por clase:")
for carpeta, cantidad in sorted(conteo_por_clase.items()):
    print(f"{carpeta}: {cantidad}")
